# LeetCode 1806 — Minimum Number of Operations to Reinitialize a Permutation bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-operations-to-reinitialize-a-permutation/ · Độ khó: **Medium** · Chủ đề: Array, Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def reinitializePermutation(self, n: int) -> int:
        ...
```

Lộ trình: **mô phỏng cả hoán vị** cho tới khi về ban đầu `O(n²)` → **chỉ theo dõi một vị trí** `O(n)` → **phân rã chu trình + BCNN** (cách tổng quát cho mọi hoán vị) → **toán: bậc của 2 theo modulo `n − 1`**. Bài này là phép **"trộn bài hoàn hảo"** (perfect out-shuffle) của bộ bài `n` lá.

## 1. Mục tiêu học

1. Mô phỏng chính xác một phép biến đổi mảng theo công thức chỉ số chẵn/lẻ.
2. Hiểu **hoán vị** như một hàm trên vị trí, và **số lần lặp để quay về** = **bậc (order)** của hoán vị = **BCNN độ dài các chu trình**.
3. Rút gọn bài toán: vị trí `j` của mỗi phần tử biến thành `2j mod (n − 1)` ⇒ đáp án là **bậc của 2 modulo `n − 1`**.
4. Nhận ra vì sao chỉ cần theo dõi **một** vị trí (vị trí 1) — và vì sao theo dõi vị trí 0 là sai.
5. Kiểm chứng các cách bằng nhau trên mọi `n` chẵn trong ràng buộc.

## 2. Đề bài & ví dụ

Cho số nguyên **chẵn** `n`. Ban đầu có hoán vị `perm` độ dài `n` với `perm[i] == i` (đánh số từ 0).

Mỗi **thao tác** tạo mảng mới `arr`, với mọi `i`:
- nếu `i % 2 == 0` thì `arr[i] = perm[i / 2]`;
- nếu `i % 2 == 1` thì `arr[i] = perm[n / 2 + (i − 1) / 2]`.

Rồi gán `perm = arr`. Trả về **số thao tác nhỏ nhất khác 0** để `perm` trở lại như ban đầu.

**Ví dụ 1:** `n = 2` → `1` (`[0,1]` → `[0,1]`).

**Ví dụ 2:** `n = 4` → `2` (`[0,1,2,3]` → `[0,2,1,3]` → `[0,1,2,3]`).

**Ví dụ 3:** `n = 6` → `4`.

**Ràng buộc:** `2 ≤ n ≤ 1000`, `n` chẵn.

In [1]:
import random
import time
from math import gcd, lcm
from typing import List

examples = [(2, 1), (4, 2), (6, 4)]
for n, want in examples:
    print(f"n={n} -> kỳ vọng {want}")

n=2 -> kỳ vọng 1
n=4 -> kỳ vọng 2
n=6 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1 Phép chia nguyên và chỉ số chẵn/lẻ

Trong Python, `i / 2` là số thực; phải dùng `i // 2`. Với `i` lẻ, `(i − 1) // 2 == i // 2`.

In [2]:
for i in range(6):
    print(i, "chẵn" if i % 2 == 0 else "lẻ ", "i//2 =", i // 2, "(i-1)//2 =", (i - 1) // 2)

0 chẵn i//2 = 0 (i-1)//2 = -1
1 lẻ  i//2 = 0 (i-1)//2 = 0
2 chẵn i//2 = 1 (i-1)//2 = 0
3 lẻ  i//2 = 1 (i-1)//2 = 1
4 chẵn i//2 = 2 (i-1)//2 = 1
5 lẻ  i//2 = 2 (i-1)//2 = 2


### 3.2 Một thao tác = "trộn bài hoàn hảo"

Chia `perm` thành nửa đầu `A = perm[:n/2]` và nửa sau `B = perm[n/2:]`, rồi xen kẽ `A[0], B[0], A[1], B[1], …`. Python viết gọn bằng gán lát cắt bước 2.

In [3]:
def one_op(perm):
    n = len(perm)
    arr = [0] * n
    arr[0::2] = perm[: n // 2]     # vị trí chẵn lấy từ nửa đầu
    arr[1::2] = perm[n // 2 :]     # vị trí lẻ lấy từ nửa sau
    return arr

print(one_op([0, 1, 2, 3, 4, 5]))

[0, 3, 1, 4, 2, 5]


### 3.3 Hoán vị, chu trình

Một hoán vị trên `{0..n−1}` tách thành các **chu trình rời nhau**: `j → f(j) → f(f(j)) → … → j`. Áp dụng hoán vị `k` lần đưa mọi phần tử về chỗ cũ ⇔ `k` chia hết cho **mọi** độ dài chu trình.

In [4]:
f = [2, 0, 1, 4, 3]          # j -> f[j]
seen, cycles = set(), []
for s in range(len(f)):
    if s in seen: continue
    c, j = [], s
    while j not in seen:
        seen.add(j); c.append(j); j = f[j]
    cycles.append(c)
print("chu trình:", cycles, "-> độ dài", [len(c) for c in cycles])

chu trình: [[0, 2, 1], [3, 4]] -> độ dài [3, 2]


### 3.4 `math.lcm` và `math.gcd`

`lcm(a, b) = a·b / gcd(a, b)`. Python ≥ 3.9 có `math.lcm(*args)` nhận nhiều đối số.

In [5]:
print(gcd(12, 18), lcm(4, 6), lcm(2, 3, 4))

6 12 12


### 3.5 Phép nhân modulo và `pow(a, k, m)`

`pow(2, k, m)` tính `2ᵏ mod m` nhanh. Bậc của `a` modulo `m` là `k > 0` nhỏ nhất với `aᵏ ≡ 1 (mod m)` (tồn tại khi `gcd(a, m) = 1`).

In [6]:
m = 9
print([pow(2, k, m) for k in range(1, 8)])

[2, 4, 8, 7, 5, 1, 2]


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng bằng tay `n = 6`:

```
ban đầu : [0, 1, 2, 3, 4, 5]
lần 1   : [0, 3, 1, 4, 2, 5]
lần 2   : [0, 4, 3, 2, 1, 5]
lần 3   : [0, 2, 4, 1, 3, 5]
lần 4   : [0, 1, 2, 3, 4, 5]   ✔ về ban đầu sau 4 lần
```

Quan sát:
- Phần tử `0` và `n − 1 = 5` **không bao giờ di chuyển** (vị trí đầu và cuối).
- Theo dõi **giá trị** `1`: nó ở vị trí `1 → 2 → 4 → 3 → 1`. Mỗi lần vị trí **nhân đôi**, rồi lấy dư cho `5 = n − 1`: `1, 2, 4, 8 mod 5 = 3, 6 mod 5 = 1`.

Vậy câu hỏi trở thành: nhân đôi bao nhiêu lần (mod `n − 1`) thì quay về 1?

In [7]:
perm = list(range(6))
for k in range(1, 5):
    perm = one_op(perm)
    print(f"lần {k}: {perm} | giá trị 1 ở vị trí {perm.index(1)}")

lần 1: [0, 3, 1, 4, 2, 5] | giá trị 1 ở vị trí 2
lần 2: [0, 4, 3, 2, 1, 5] | giá trị 1 ở vị trí 4
lần 3: [0, 2, 4, 1, 3, 5] | giá trị 1 ở vị trí 3
lần 4: [0, 1, 2, 3, 4, 5] | giá trị 1 ở vị trí 1


## 5. Cách 1 — Brute Force / baseline: mô phỏng cả hoán vị

**Ý tưởng:** lặp thao tác, sau mỗi lần so `perm` với hoán vị đơn vị.

In [8]:
def reinit_simulate(n):
    target = list(range(n))
    perm = target[:]
    ops = 0
    while True:
        perm = [perm[i // 2] if i % 2 == 0 else perm[n // 2 + (i - 1) // 2] for i in range(n)]
        ops += 1
        if perm == target:
            return ops

for n, want in examples:
    print(n, "->", reinit_simulate(n), "| kỳ vọng", want)

2 -> 1 | kỳ vọng 1
4 -> 2 | kỳ vọng 2
6 -> 4 | kỳ vọng 4


**Độ phức tạp:** mỗi thao tác `O(n)`; số thao tác là đáp án `A`. Ta sẽ chứng minh `A ≤ n − 2` (với `n ≥ 4`) ⇒ `O(n²)` = `10⁶` với `n = 1000` — **đủ nhanh** để nộp. Nhưng vẫn lãng phí: ta biến đổi `n` phần tử trong khi chỉ cần biết một chu trình.

**Nút thắt:** so sánh và tạo lại cả mảng mỗi lần.

In [9]:
t = time.perf_counter()
mx = max((reinit_simulate(n), n) for n in range(2, 1001, 2))
print("đáp án lớn nhất với n ≤ 1000:", mx, f"| tổng thời gian mô phỏng mọi n: {time.perf_counter()-t:.2f}s")

đáp án lớn nhất với n ≤ 1000: (946, 948) | tổng thời gian mô phỏng mọi n: 3.01s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — công thức di chuyển.** Giá trị ở vị trí `j` sau một thao tác sẽ đến vị trí:
- `2j` nếu `j < n/2` (vì `arr[2j] = perm[j]`);
- `2(j − n/2) + 1 = 2j − (n − 1)` nếu `j ≥ n/2` (vì `arr[2t+1] = perm[n/2 + t]`).

Với `0 ≤ j ≤ n − 2`, cả hai trường hợp gộp lại: **`j ↦ 2j mod (n − 1)`**. Riêng `j = n − 1` cố định.

**Insight 2 — bậc của hoán vị.** Hoán vị quay về đơn vị sau `k` lần ⇔ `2ᵏ · j ≡ j (mod n − 1)` với mọi `j` ⇔ (lấy `j = 1`) `2ᵏ ≡ 1 (mod n − 1)`. Ngược lại nếu `2ᵏ ≡ 1` thì đúng với mọi `j`.

**Quy tắc:** đáp án = số `k > 0` nhỏ nhất với `2ᵏ ≡ 1 (mod n − 1)`; với `n = 2` thì `n − 1 = 1`, mọi số ≡ 0 ≡ 1 (mod 1) ⇒ đáp án 1.

**Hệ quả:** chỉ cần theo dõi **vị trí 1**: `i = 1; lặp i = 2i mod (n−1)` cho tới khi `i == 1`. Vì `n − 1` lẻ nên `gcd(2, n − 1) = 1`, bậc luôn tồn tại và ≤ `φ(n−1) ≤ n − 2`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chỉ theo dõi một vị trí (`O(A)` ≤ `O(n)`)

Không cần công thức toán: dùng đúng định nghĩa thao tác để tìm vị trí mới của phần tử đang ở vị trí `j`.

In [10]:
def next_pos(j, n):
    """Phần tử đang ở vị trí j sẽ tới vị trí nào sau một thao tác."""
    return 2 * j if j < n // 2 else 2 * (j - n // 2) + 1

def reinit_track(n):
    j, ops = 1, 0
    while True:
        j = next_pos(j, n)
        ops += 1
        if j == 1:
            return ops

for n, want in examples:
    print(n, "->", reinit_track(n), "| kỳ vọng", want)
print("next_pos với n=6:", [next_pos(j, 6) for j in range(6)])

2 -> 1 | kỳ vọng 1
4 -> 2 | kỳ vọng 2
6 -> 4 | kỳ vọng 4
next_pos với n=6: [0, 2, 4, 1, 3, 5]


**Độ phức tạp:** `O(A)` thời gian, `O(1)` bộ nhớ. Vì sao vị trí 1 đủ đại diện? Theo Insight 2, chu trình chứa vị trí 1 có độ dài đúng bằng bậc của 2, và mọi chu trình khác có độ dài **chia hết** số đó.

### 7.2 Cách 3 — Phân rã chu trình + BCNN (tổng quát cho mọi hoán vị)

Nếu quên mất cấu trúc "nhân đôi", vẫn có cách an toàn: tính hàm vị trí `next_pos` cho mọi `j`, tách chu trình, đáp án là BCNN độ dài các chu trình.

In [11]:
def permutation_order(f):
    seen = [False] * len(f)
    ans = 1
    for s in range(len(f)):
        if seen[s]:
            continue
        L, j = 0, s
        while not seen[j]:
            seen[j] = True
            j = f[j]
            L += 1
        ans = lcm(ans, L)
    return ans

class SolutionCycles:
    def reinitializePermutation(self, n: int) -> int:
        return permutation_order([next_pos(j, n) for j in range(n)])

for n, want in examples:
    print(n, "->", SolutionCycles().reinitializePermutation(n), "| kỳ vọng", want)

2 -> 1 | kỳ vọng 1
4 -> 2 | kỳ vọng 2
6 -> 4 | kỳ vọng 4


**Độ phức tạp:** `O(n)` (mỗi vị trí thăm một lần) + chi phí `lcm`. Đây là kỹ thuật dùng được cho **mọi** phép xáo trộn cố định, không riêng bài này.

### 7.3 Cách 4 — Bậc của 2 modulo `n − 1`

In [12]:
class SolutionOrder:
    def reinitializePermutation(self, n: int) -> int:
        m = n - 1
        k, x = 1, 2 % m
        while x != 1 % m:          # với m = 1: 1 % 1 == 0 == 2 % 1 -> k = 1
            x = x * 2 % m
            k += 1
        return k

for n, want in examples:
    print(n, "->", SolutionOrder().reinitializePermutation(n), "| kỳ vọng", want)

2 -> 1 | kỳ vọng 1
4 -> 2 | kỳ vọng 2
6 -> 4 | kỳ vọng 4


**Độ phức tạp:** `O(A)` thời gian, `O(1)` bộ nhớ — giống Cách 2 nhưng phép tính gọn hơn (chỉ nhân đôi và lấy dư). Có thể tăng tốc lý thuyết bằng phân tích `φ(n−1)` thành thừa số và thử ước, nhưng với `n ≤ 1000` không cần.

## 8. TRACE từng bước

Trace Cách 2/4 với `n = 6` (`m = n − 1 = 5`) và `n = 10` (`m = 9`):

| bước | `n = 6`: vị trí `j` | `2j mod 5` | `n = 10`: vị trí `j` | `2j mod 9` |
|---|---|---|---|---|
| 0 | 1 | — | 1 | — |
| 1 | 2 | 2 | 2 | 2 |
| 2 | 4 | 4 | 4 | 4 |
| 3 | 3 | 8 mod 5 = 3 | 8 | 8 |
| 4 | **1** | 6 mod 5 = 1 | 7 | 16 mod 9 = 7 |
| 5 | | | 5 | 14 mod 9 = 5 |
| 6 | | | **1** | 10 mod 9 = 1 |

Đáp án: `n = 6` → 4, `n = 10` → 6.

In [13]:
def trace_track(n):
    j, ops = 1, 0
    print(f"n={n}: bắt đầu vị trí 1")
    while True:
        nj = next_pos(j, n)
        ops += 1
        nhanh = "nửa đầu -> 2j" if j < n // 2 else "nửa sau -> 2(j-n/2)+1"
        print(f"  bước {ops}: j={j} ({nhanh}) -> {nj} | kiểm tra 2j mod (n-1) = {2 * j % (n - 1) if n > 2 else 0}")
        j = nj
        if j == 1:
            print(f"  => quay về 1 sau {ops} bước")
            return ops

trace_track(6)
trace_track(10)

n=6: bắt đầu vị trí 1
  bước 1: j=1 (nửa đầu -> 2j) -> 2 | kiểm tra 2j mod (n-1) = 2
  bước 2: j=2 (nửa đầu -> 2j) -> 4 | kiểm tra 2j mod (n-1) = 4
  bước 3: j=4 (nửa sau -> 2(j-n/2)+1) -> 3 | kiểm tra 2j mod (n-1) = 3
  bước 4: j=3 (nửa sau -> 2(j-n/2)+1) -> 1 | kiểm tra 2j mod (n-1) = 1
  => quay về 1 sau 4 bước
n=10: bắt đầu vị trí 1
  bước 1: j=1 (nửa đầu -> 2j) -> 2 | kiểm tra 2j mod (n-1) = 2
  bước 2: j=2 (nửa đầu -> 2j) -> 4 | kiểm tra 2j mod (n-1) = 4
  bước 3: j=4 (nửa đầu -> 2j) -> 8 | kiểm tra 2j mod (n-1) = 8
  bước 4: j=8 (nửa sau -> 2(j-n/2)+1) -> 7 | kiểm tra 2j mod (n-1) = 7
  bước 5: j=7 (nửa sau -> 2(j-n/2)+1) -> 5 | kiểm tra 2j mod (n-1) = 5
  bước 6: j=5 (nửa sau -> 2(j-n/2)+1) -> 1 | kiểm tra 2j mod (n-1) = 1
  => quay về 1 sau 6 bước


6

Trace Cách 3 (chu trình) với `n = 10` để thấy mọi chu trình có độ dài chia hết 6:

In [14]:
def trace_cycles(n):
    f = [next_pos(j, n) for j in range(n)]
    seen, out = set(), []
    for s in range(n):
        if s in seen: continue
        c, j = [], s
        while j not in seen:
            seen.add(j); c.append(j); j = f[j]
        out.append(c)
    for c in out:
        print(f"  chu trình {c} (độ dài {len(c)})")
    print("  BCNN =", lcm(*[len(c) for c in out]))

trace_cycles(10)

  chu trình [0] (độ dài 1)
  chu trình [1, 2, 4, 8, 7, 5] (độ dài 6)
  chu trình [3, 6] (độ dài 2)
  chu trình [9] (độ dài 1)
  BCNN = 6


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def reinitializePermutation(self, n: int) -> int:
        # vị trí j (0 < j < n-1) biến thành 2j mod (n-1); theo dõi vị trí 1
        j, ops = 1, 0
        while True:
            j = 2 * j if j < n // 2 else 2 * j - (n - 1)
            ops += 1
            if j == 1:
                return ops

for n, want in examples:
    assert Solution().reinitializePermutation(n) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


Với `n = 2`: `j = 1 ≥ n//2 = 1` nên `j = 2 − 1 = 1` ngay ⇒ trả 1. ✔

## 10. Vì sao thuật toán đúng

**Bổ đề 1 (công thức vị trí):** sau một thao tác, giá trị ở vị trí `j` chuyển tới `g(j)`, với `g(j) = 2j` nếu `j < n/2` và `g(j) = 2j − (n − 1)` nếu `j ≥ n/2`. *Chứng minh:* từ định nghĩa, `arr[2t] = perm[t]` (với `t < n/2`) và `arr[2t+1] = perm[n/2 + t]`; đặt `j = t` hoặc `j = n/2 + t`.

**Bổ đề 2:** với `0 ≤ j ≤ n − 2`, `g(j) ≡ 2j (mod n − 1)` và `g(j) ∈ [0, n − 2]`; `g(n − 1) = n − 1`.

**Định lý:** `perm` về đơn vị sau `k` thao tác ⇔ `gᵏ(j) = j` với mọi `j` ⇔ `2ᵏ j ≡ j (mod n − 1)` với mọi `j ∈ [0, n−2]` ⇔ `2ᵏ ≡ 1 (mod n − 1)` (chiều ⇐: nhân hai vế với `j`; chiều ⇒: chọn `j = 1`).

Do đó đáp án là bậc của 2 modulo `n − 1` — đúng thứ Cách 2, 4 và lời giải nộp tính (chúng lặp `j ↦ g(j)` từ `j = 1` tới khi quay lại 1). Vì `n − 1` lẻ, `gcd(2, n−1) = 1`, nên theo định lý Euler `2^{φ(n−1)} ≡ 1` — vòng lặp chắc chắn dừng sau ≤ `φ(n−1) ≤ n − 2` bước (với `n ≥ 4`).

## 11. Phân tích Time / Space

Gọi `A` = đáp án (≤ `n − 2` với `n ≥ 4`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng cả mảng | `O(n · A)` ⊆ `O(n²)` | `O(n)` |
| 2. Theo dõi vị trí 1 | `O(A)` ⊆ `O(n)` | `O(1)` |
| 3. Chu trình + BCNN | `O(n)` | `O(n)` |
| 4. Bậc của 2 mod `n−1` | `O(A)` | `O(1)` |

In [16]:
for name, f in [("Cách 1", reinit_simulate), ("Cách 2", reinit_track),
                ("Cách 3", SolutionCycles().reinitializePermutation),
                ("Cách 4", SolutionOrder().reinitializePermutation),
                ("Nộp", Solution().reinitializePermutation)]:
    t = time.perf_counter()
    for n in range(2, 1001, 2):
        f(n)
    print(f"{name}: mọi n chẵn 2..1000 trong {1000*(time.perf_counter()-t):.1f} ms")

Cách 1: mọi n chẵn 2..1000 trong 2922.7 ms
Cách 2: mọi n chẵn 2..1000 trong 6.2 ms
Cách 3: mọi n chẵn 2..1000 trong 25.1 ms
Cách 4: mọi n chẵn 2..1000 trong 3.8 ms
Nộp: mọi n chẵn 2..1000 trong 4.3 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
sols = [Solution(), SolutionCycles(), SolutionOrder()]
edge = examples + [(8, 3), (10, 6), (12, 10), (1000, 36)]
for n, want in edge:
    for s in sols:
        assert s.reinitializePermutation(n) == want, (n, s)
    assert reinit_simulate(n) == want == reinit_track(n)
    print("OK", n, "->", want)

OK 2 -> 1
OK 4 -> 2
OK 6 -> 4
OK 8 -> 3
OK 10 -> 6
OK 12 -> 10
OK 1000 -> 36


(Các giá trị `n = 8, 10, 12, 1000` không có trong đề — đã đối chiếu bằng mô phỏng trực tiếp ở dòng `assert reinit_simulate(n) == want`.)

### 12.2 Kiểm tra **toàn bộ** miền ràng buộc

Miền đầu vào chỉ có 500 giá trị (`n` chẵn từ 2 tới 1000) — kiểm hết, không cần ngẫu nhiên.

In [18]:
for n in range(2, 1001, 2):
    want = reinit_simulate(n)
    assert reinit_track(n) == want
    for s in sols:
        assert s.reinitializePermutation(n) == want, n
print("mọi n chẵn 2..1000: 4 cách khớp mô phỏng")

mọi n chẵn 2..1000: 4 cách khớp mô phỏng


### 12.3 Kiểm tra tính chất: mọi chu trình có độ dài chia hết đáp án, đáp án ≤ n − 2

In [19]:
rng = random.Random(1806)
for n in rng.sample(range(4, 1001, 2), 60):
    f = [next_pos(j, n) for j in range(n)]
    A = Solution().reinitializePermutation(n)
    assert A <= n - 2
    assert pow(2, A, n - 1) == 1
    seen = [False] * n
    for s in range(n):
        if seen[s]: continue
        L, j = 0, s
        while not seen[j]:
            seen[j] = True; j = f[j]; L += 1
        assert A % L == 0
print("tính chất đúng trên 60 giá trị n ngẫu nhiên (seed 1806)")

tính chất đúng trên 60 giá trị n ngẫu nhiên (seed 1806)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra "đã về ban đầu" **trước** khi thực hiện thao tác

In [20]:
def wrong_check_first(n):
    target = list(range(n)); perm = target[:]; ops = 0
    while perm != target:                  # SAI: điều kiện đã sai ngay từ đầu -> không lặp
        perm = one_op(perm); ops += 1
    return ops

print("sai:", [wrong_check_first(n) for n in (2, 4, 6)], "| đúng:", [Solution().reinitializePermutation(n) for n in (2, 4, 6)])

sai: [0, 0, 0] | đúng: [1, 2, 4]


**Vì sao sai:** đề hỏi số thao tác **khác 0**; trạng thái ban đầu đã là đơn vị nên vòng lặp không chạy lần nào. **Sửa:** dùng `while True` — làm thao tác trước, kiểm sau.

### Lỗi 2 — Lấy dư cho `n` thay vì `n − 1`

In [21]:
def wrong_mod_n(n, cap=5000):
    j, ops = 1, 0
    while True:
        j = 2 * j % n                       # SAI: modulo n
        ops += 1
        if j == 1:
            return ops
        if ops > cap:
            return f"không dừng sau {cap} bước"

for n in (4, 6, 10):
    print(n, "sai:", wrong_mod_n(n), "| đúng:", Solution().reinitializePermutation(n))

4 sai: không dừng sau 5000 bước | đúng: 2
6 sai: không dừng sau 5000 bước | đúng: 4
10 sai: không dừng sau 5000 bước | đúng: 6


**Vì sao sai:** `n` chẵn nên `2j mod n` luôn chẵn — không bao giờ quay về 1 ⇒ vòng lặp vô hạn (ta chặn bằng `cap` để demo). Vị trí cuối `n − 1` cố định, phần còn lại là `n − 1` vị trí `0..n−2` ⇒ modulo `n − 1`.

### Lỗi 3 — Theo dõi vị trí 0 (hoặc `n − 1`)

In [22]:
def wrong_track_zero(n):
    j, ops = 0, 0
    while True:
        j = next_pos(j, n); ops += 1
        if j == 0:
            return ops

print("sai:", [wrong_track_zero(n) for n in (4, 6, 10)], "| đúng:", [Solution().reinitializePermutation(n) for n in (4, 6, 10)])

sai: [1, 1, 1] | đúng: [2, 4, 6]


**Vì sao sai:** vị trí 0 là **điểm bất động** (chu trình độ dài 1) nên luôn "quay về" sau 1 bước. Phải theo dõi một vị trí nằm trên chu trình **dài nhất** — vị trí 1 (Insight 2).

### Lỗi 4 — Với hoán vị tổng quát, lấy **max** độ dài chu trình thay vì **BCNN**

In [23]:
def wrong_order_max(f):
    seen = [False] * len(f); best = 1
    for s in range(len(f)):
        if seen[s]: continue
        L, j = 0, s
        while not seen[j]:
            seen[j] = True; j = f[j]; L += 1
        best = max(best, L)                 # SAI cho hoán vị tổng quát
    return best

f = [1, 0, 3, 4, 2]                         # chu trình (0 1) và (2 3 4)
print("sai:", wrong_order_max(f), "| đúng:", permutation_order(f))
def apply_k(f, k):
    pos = list(range(len(f)))
    for _ in range(k):
        pos = [f[x] for x in pos]
    return pos

for k in (3, 6):
    print(f"áp dụng {k} lần: {apply_k(f, k)} | về đơn vị: {apply_k(f, k) == list(range(5))}")

sai: 3 | đúng: 6
áp dụng 3 lần: [1, 0, 2, 3, 4] | về đơn vị: False
áp dụng 6 lần: [0, 1, 2, 3, 4] | về đơn vị: True


**Vì sao sai:** chu trình dài 2 và 3 chỉ cùng về chỗ sau `lcm(2, 3) = 6` lần, không phải 3. Ở **bài này** max tình cờ bằng BCNN (mọi độ dài chia hết độ dài chu trình chứa 1), nên lỗi chỉ lộ khi áp dụng kỹ thuật cho phép xáo trộn khác. **Sửa:** dùng `lcm`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Lặp một phép xáo trộn cố định, bao lâu thì quay về?"** ⇒ bậc của hoán vị = BCNN độ dài chu trình.
- Viết hàm **`next_pos(j)`** (phần tử ở `j` đi đâu) thay vì biến đổi cả mảng.
- Nếu `next_pos` có dạng `a·j mod m` ⇒ đáp án là **bậc của `a` modulo `m`**.
- Cẩn thận **điểm bất động** (ở đây 0 và `n − 1`) — đừng chọn chúng để theo dõi.
- Ràng buộc nhỏ (`n ≤ 1000`) ⇒ mô phỏng `O(n²)` cũng qua; nhưng hiểu cấu trúc giúp giải biến thể `n` lớn.

## 15. Biến thể của bài

1. **In-shuffle** (xen kẽ bắt đầu bằng nửa **sau**): vị trí `j ↦ 2(j+1) mod (n+1) − 1` ⇒ đáp án là bậc của 2 modulo `n + 1` (Bài tập 5).
2. **`n` lẻ** hoặc chia bài không đều: vẫn dùng Cách 3 (chu trình + BCNN).
3. **`n` tới `10¹⁸`:** tính bậc của 2 mod `n − 1` bằng phân tích thừa số `φ(n−1)` và thử các ước với `pow(2, d, m)`.
4. **Hỏi trạng thái sau `k` thao tác** (`k` lớn): mỗi vị trí `j ↦ 2ᵏ j mod (n−1)` — `pow(2, k, n-1)`.
5. Bài cùng ý tưởng chu trình: LeetCode 1914 *Cyclically Rotating a Grid*, 957 *Prison Cells After N Days* (tìm chu kỳ trạng thái).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | biến đổi cả mảng tới khi về đơn vị | `O(n²)` | `O(n)` | kiểm chứng; `n` nhỏ |
| 2. Theo dõi vị trí 1 | lặp `next_pos` | `O(n)` | `O(1)` | **nộp bài**, không cần toán |
| 3. Chu trình + BCNN | bậc hoán vị tổng quát | `O(n)` | `O(n)` | mọi phép xáo trộn |
| 4. Bậc của 2 mod `n−1` | toán số học | `O(n)` | `O(1)` | hiểu bản chất, mở rộng `n` lớn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `reinitializePermutation(8)` và `reinitializePermutation(14)`?

**Bài 2 (tính tay):** Với `n = 10`, viết dãy vị trí của giá trị 1 qua từng thao tác cho tới khi quay về.

**Bài 3 (điền TODO):** Viết `cycle_lengths(f)` trả về list độ dài các chu trình của hoán vị `f` (theo thứ tự gặp).

**Bài 4 (tự viết lại):** Viết `multiplicative_order(a, m)` — bậc của `a` modulo `m` (giả sử `gcd(a, m) = 1`, `m ≥ 1`), rồi dùng nó viết lời giải một dòng.

**Bài 5 (mini challenge — in-shuffle):** Thao tác mới: `arr[i] = perm[n/2 + i/2]` với `i` chẵn, `arr[i] = perm[(i−1)/2]` với `i` lẻ (nửa sau lên trước). Viết `reinit_in_shuffle(n)` bằng toán và kiểm chứng bằng mô phỏng.

In [24]:
def cycle_lengths(f):
    # TODO
    pass

def multiplicative_order(a, m):
    # TODO
    pass

def reinit_in_shuffle(n):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Bậc của 2 mod `n − 1` | `n = 8`: mod 7: 2, 4, 1 | `n = 14`: mod 13 — 2 là căn nguyên thủy |
| 2 | Nhân đôi mod 9 | 1, 2, 4, 8, 7, 5, 1 | 6 bước |
| 3 | Mảng `seen` | Đi theo `f` tới khi gặp nút đã thăm | Đếm số bước |
| 4 | `x = a % m`, `k = 1` | Lặp `x = x * a % m` tới khi `x == 1 % m` | `m = 1` trả 1 |
| 5 | Viết `next_pos` mới: `j < n/2 → 2j + 1`, ngược lại `2(j − n/2)` | Đặt `p = j + 1`: `p ↦ 2p mod (n+1)` | Đáp án `multiplicative_order(2, n + 1)` |

## 19. Đáp án tham khảo

In [25]:
print("Bài 1:", Solution().reinitializePermutation(8), Solution().reinitializePermutation(14))
assert Solution().reinitializePermutation(8) == 3 and Solution().reinitializePermutation(14) == 12
seq, j = [1], 1
while True:
    j = next_pos(j, 10); seq.append(j)
    if j == 1: break
print("Bài 2:", seq); assert seq == [1, 2, 4, 8, 7, 5, 1]

Bài 1: 3 12
Bài 2: [1, 2, 4, 8, 7, 5, 1]


In [26]:
def cycle_lengths(f):
    seen = [False] * len(f); out = []
    for s in range(len(f)):
        if seen[s]: continue
        L, j = 0, s
        while not seen[j]:
            seen[j] = True; j = f[j]; L += 1
        out.append(L)
    return out

def multiplicative_order(a, m):
    k, x = 1, a % m
    while x != 1 % m:
        x = x * a % m
        k += 1
    return k

assert cycle_lengths([1, 0, 3, 4, 2]) == [2, 3]
for n in range(2, 1001, 2):
    assert multiplicative_order(2, n - 1) == Solution().reinitializePermutation(n)
    assert lcm(*cycle_lengths([next_pos(j, n) for j in range(n)])) == Solution().reinitializePermutation(n)
print("Bài 3, 4: OK", cycle_lengths([next_pos(j, 10) for j in range(10)]))

Bài 3, 4: OK [1, 6, 2, 1]


In [27]:
def in_shuffle_op(perm):
    n = len(perm)
    return [perm[n // 2 + i // 2] if i % 2 == 0 else perm[(i - 1) // 2] for i in range(n)]

def in_shuffle_simulate(n):
    target = list(range(n)); perm = target[:]; ops = 0
    while True:
        perm = in_shuffle_op(perm); ops += 1
        if perm == target:
            return ops

def reinit_in_shuffle(n):
    return multiplicative_order(2, n + 1)

for n in range(2, 401, 2):
    assert reinit_in_shuffle(n) == in_shuffle_simulate(n), n
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK; ví dụ n=4,6,8 ->",
      [reinit_in_shuffle(n) for n in (4, 6, 8)])

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK; ví dụ n=4,6,8 -> [4, 3, 6]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# phần tử ở vị trí j (0 < j < n-1) tới 2j mod (n-1)
j, ops = 1, 0
while True:
    j = 2 * j if j < n // 2 else 2 * j - (n - 1)
    ops += 1
    if j == 1:
        return ops
```

**Checklist:**
- [ ] Làm thao tác **trước**, kiểm tra **sau** (đáp án khác 0)?
- [ ] Modulo `n − 1`, không phải `n`?
- [ ] Theo dõi vị trí **1**, không phải điểm bất động 0 / `n − 1`?
- [ ] `n = 2` trả 1?
- [ ] Nếu dùng chu trình: BCNN, không phải max.